# Stage 5 - Gold Analytics

Aggregates `workspace.default.capstone_silver_sales` into the Gold summary table:
`workspace.default.capstone_gold_sales_summary`

Metrics calculated per `year`, `month`, `month_name`, `state`, `category`:

| Metric | Description |
|--------|-------------|
| total_orders | COUNT of distinct order_id |
| units_sold | SUM of quantity |
| gross_sales | SUM of gross_amount |
| total_discount | SUM of discount_amount |
| net_sales | SUM of net_amount |
| avg_order_value | AVG net_amount per order |

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

## 1. Configuration

In [0]:
SILVER_TABLE = "workspace.default.capstone_silver_sales"
GOLD_TABLE   = "workspace.default.capstone_gold_sales_summary"

## 2. Load Silver

In [0]:
df_silver = spark.read.table(SILVER_TABLE)
print(f"Silver row count: {df_silver.count()}")

## 3. Aggregate to Gold

In [0]:
df_gold = (
    df_silver
    .groupBy("year", "month", "month_name", "state", "category")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("gross_amount"), 2).alias("gross_sales"),
        F.round(F.sum("discount_amount"), 2).alias("total_discount"),
        F.round(F.sum("net_amount"), 2).alias("net_sales"),
        F.round(F.avg("net_amount"), 2).alias("avg_order_value"),
    )
    .orderBy("year", "month", "state", "category")
)

print(f"Gold summary rows: {df_gold.count()}")

## 4. Write Gold table

In [0]:
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

print(f"Gold table written: {GOLD_TABLE}")

## 5. Dashboard-ready queries

Run the cells below in Databricks SQL or paste the equivalent SQL into a dashboard.

In [0]:
gold = spark.read.table(GOLD_TABLE)

In [0]:
# Chart 1 - Monthly Sales (net_sales by year-month)
print("=== Chart 1: Monthly Net Sales ===")
(
    gold
    .groupBy("year", "month", "month_name")
    .agg(F.round(F.sum("net_sales"), 2).alias("monthly_net_sales"))
    .orderBy("year", "month")
    .show(24, truncate=False)
)

In [0]:
# Chart 2 - Sales by State
print("=== Chart 2: Net Sales by State ===")
(
    gold
    .groupBy("state")
    .agg(F.round(F.sum("net_sales"), 2).alias("state_net_sales"))
    .orderBy(F.desc("state_net_sales"))
    .show(truncate=False)
)

In [0]:
# Chart 3 - Sales by Category
print("=== Chart 3: Net Sales by Category ===")
(
    gold
    .groupBy("category")
    .agg(F.round(F.sum("net_sales"), 2).alias("category_net_sales"))
    .orderBy(F.desc("category_net_sales"))
    .show(truncate=False)
)

In [0]:
# Chart 4 - Orders by Month
print("=== Chart 4: Total Orders by Month ===")
(
    gold
    .groupBy("year", "month", "month_name")
    .agg(F.sum("total_orders").alias("orders_per_month"))
    .orderBy("year", "month")
    .show(24, truncate=False)
)

## 6. Overall KPIs

In [0]:
kpi = gold.agg(
    F.sum("total_orders").alias("total_orders"),
    F.sum("units_sold").alias("units_sold"),
    F.round(F.sum("gross_sales"), 2).alias("gross_sales"),
    F.round(F.sum("total_discount"), 2).alias("total_discount"),
    F.round(F.sum("net_sales"), 2).alias("net_sales"),
    F.round(F.sum("net_sales") / F.sum("total_orders"), 2).alias("avg_order_value"),
)

print("=== Overall KPIs ===")
kpi.show(truncate=False)

In [0]:
# %sql
# DROP TABLE IF EXISTS workspace.default.capstone_staging_sales;
# DROP TABLE IF EXISTS workspace.default.capstone_bronze_sales;
# DROP TABLE IF EXISTS workspace.default.capstone_silver_sales;
# DROP TABLE IF EXISTS workspace.default.capstone_gold_sales_summary;
# DROP TABLE IF EXISTS workspace.default.capstone_dq_summary;